# SmartServe Food Intelligence
## 01 — Data Understanding

### Objective

This notebook provides an initial understanding of the SmartServe
Food Intelligence dataset before data quality checks, exploratory
analysis, SQL analysis, and machine learning.

The analysis covers:

- Dataset structure
- Dataset size
- Column information
- Data types
- Date coverage
- Dimension characteristics
- Fact-table characteristics
- Basic business-level observations

No data is modified in this notebook.

### Import libraries and project modules

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_all_data

In [2]:
data = load_all_data()

print(f"Datasets loaded: {len(data)}")

2026-11-07 11:59:34,966 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_date.csv
2026-11-07 11:59:35,135 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_store.csv
2026-11-07 11:59:35,146 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_product.csv
2026-11-07 11:59:35,151 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_customer.csv
2026-11-07 11:59:35,187 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_promotion.csv
2026-11-07 11:59:35,202 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartServe-Food-Intelligence\data\raw\dim_holiday.csv
2026-11-07 11:59:35,210 | INFO | src.data_loader | Loading data file: D:\Data Analysis Projects\SmartS

Datasets loaded: 11


### 1. Dataset Overview

SmartServe uses a star-schema-oriented analytical structure containing
dimension tables and fact tables.

Dimension tables describe business entities such as dates, stores,
products, customers, promotions, holidays, and weather.

Fact tables capture operational events and measurements such as sales,
preparation, inventory, and waste.

In [3]:
overview = []

for name, df in data.items():
    overview.append(
        {
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "memory_mb": round(
                df.memory_usage(deep=True).sum() / 1024**2,
                2,
            ),
        }
    )

overview_df = pd.DataFrame(overview)

overview_df

,dataset,rows,columns,memory_mb
0,dim_date,90,14,0.03
1,dim_store,5,9,0.00
2,dim_product,20,10,0.01
3,dim_customer,1000,6,0.27
4,dim_promotion,8,7,0.00
5,dim_holiday,8,4,0.00
6,dim_weather,450,7,0.09
7,fact_sales,122318,15,54.05
8,fact_preparation,9000,8,1.80
9,fact_inventory,9000,12,2.01


### Examine columns and data types

In [4]:
for name, df in data.items():

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    print(
        pd.DataFrame(
            {
                "column": df.columns,
                "dtype": df.dtypes.astype(str),
                "missing": df.isna().sum().values,
                "unique": [
                    df[column].nunique()
                    for column in df.columns
                ],
            }
        ).to_string(index=False)
    )


DIM_DATE
      column   dtype  missing  unique
     date_id   int64        0      90
   full_date     str        0      90
        year   int64        0       1
     quarter   int64        0       1
       month   int64        0       3
  month_name     str        0       3
        week   int64        0      14
         day   int64        0      31
    day_name     str        0       7
 day_of_week   int64        0       7
  is_weekend    bool        0       2
  is_holiday    bool        0       1
holiday_name float64       90       0
      season     str        0       2

DIM_STORE
                 column dtype  missing  unique
               store_id   str        0       5
             store_name   str        0       5
                   city   str        0       5
                   area   str        0       5
             store_type   str        0       3
           opening_date   str        0       1
       seating_capacity int64        0       5
average_daily_customers int64    

### Examine date coverage

In [5]:
dates = data["dim_date"].copy()

dates["full_date"] = pd.to_datetime(
    dates["full_date"]
)

print(
    "Minimum Date:",
    dates["full_date"].min()
)

print(
    "Maximum Date:",
    dates["full_date"].max()
)

print(
    "Number of Dates:",
    dates["full_date"].nunique()
)

Minimum Date: 2025-01-01 00:00:00
Maximum Date: 2025-03-31 00:00:00
Number of Dates: 90


### 2. Dimension Analysis

The dimension tables provide descriptive context for the operational
facts.

In [6]:
dimension_tables = [
    "dim_store",
    "dim_product",
    "dim_customer",
    "dim_promotion",
    "dim_holiday",
    "dim_weather",
]

for name in dimension_tables:

    df = data[name]

    print(
        f"{name:<20} "
        f"Rows: {len(df):>6,} | "
        f"Columns: {len(df.columns):>2}"
    )

dim_store            Rows:      5 | Columns:  9
dim_product          Rows:     20 | Columns: 10
dim_customer         Rows:  1,000 | Columns:  6
dim_promotion        Rows:      8 | Columns:  7
dim_holiday          Rows:      8 | Columns:  4
dim_weather          Rows:    450 | Columns:  7


### Store overview

In [7]:
stores = data["dim_store"]

print("Number of stores:", stores["store_id"].nunique())

print("\nStores by city:")
print(
    stores["city"]
    .value_counts()
)

print("\nStores by type:")
print(
    stores["store_type"]
    .value_counts()
)

print("\nStores by region:")
print(
    stores["manager_region"]
    .value_counts()
)

Number of stores: 5

Stores by city:
city
Dhaka         1
Chattogram    1
Sylhet        1
Rajshahi      1
Khulna        1
Name: count, dtype: int64

Stores by type:
store_type
Flagship    2
Standard    2
Express     1
Name: count, dtype: int64

Stores by region:
manager_region
Central    1
East       1
North      1
South      1
West       1
Name: count, dtype: int64


### Product overview

In [8]:
products = data["dim_product"]

print(
    "Number of products:",
    products["product_id"].nunique(),
)

print("\nProducts by category:")
print(
    products["category"]
    .value_counts()
)

print("\nProducts by subcategory:")
print(
    products["subcategory"]
    .value_counts()
)

print("\nAverage selling price by category:")
print(
    products.groupby("category")["selling_price"]
    .mean()
    .round(2)
)

Number of products: 20

Products by category:
category
Beverage    10
Food         5
Dessert      5
Name: count, dtype: int64

Products by subcategory:
subcategory
Coffee         5
Cold Drinks    3
Bakery         3
Sandwich       2
Burger         2
Cake           2
Smoothie       1
Shake          1
Sides          1
Name: count, dtype: int64

Average selling price by category:
category
Beverage    172.0
Dessert     172.0
Food        270.0
Name: selling_price, dtype: float64


### Customer overview

In [9]:
customers = data["dim_customer"]

print(
    "Number of customers:",
    customers["customer_id"].nunique(),
)

print("\nCustomer segments:")
print(
    customers["customer_segment"]
    .value_counts()
)

print("\nAge groups:")
print(
    customers["age_group"]
    .value_counts()
)

print("\nAcquisition channels:")
print(
    customers["acquisition_channel"]
    .value_counts()
)

print("\nLoyalty membership:")
print(
    customers["loyalty_member"]
    .value_counts()
)

Number of customers: 1000

Customer segments:
customer_segment
Regular       437
Occasional    307
Premium       150
New           106
Name: count, dtype: int64

Age groups:
age_group
55+      224
45-54    219
25-34    196
35-44    192
18-24    169
Name: count, dtype: int64

Acquisition channels:
acquisition_channel
Referral        212
Website         202
Walk-in         200
Mobile App      199
Social Media    187
Name: count, dtype: int64

Loyalty membership:
loyalty_member
False    595
True     405
Name: count, dtype: int64


### Fact-table overview

In [10]:
fact_tables = [
    "fact_sales",
    "fact_preparation",
    "fact_inventory",
    "fact_waste",
]

for name in fact_tables:

    df = data[name]

    print(
        f"\n{name.upper()}"
    )

    print(
        f"Rows: {len(df):,}"
    )

    print(
        f"Columns: {len(df.columns)}"
    )


FACT_SALES
Rows: 122,318
Columns: 15

FACT_PREPARATION
Rows: 9,000
Columns: 8

FACT_INVENTORY
Rows: 9,000
Columns: 12

FACT_WASTE
Rows: 3,648
Columns: 8


### Sales overview

In [11]:
sales = data["fact_sales"]

print(
    "Total quantity sold:",
    sales["quantity"].sum()
)

print(
    "Unique orders:",
    sales["order_id"].nunique()
)

print(
    "Unique customers:",
    sales["customer_id"].nunique()
)

print(
    "Total gross sales:",
    round(
        sales["gross_sales"].sum(),
        2,
    )
)

print(
    "Total net sales:",
    round(
        sales["net_sales"].sum(),
        2,
    )
)

print(
    "Total profit:",
    round(
        sales["profit_amount"].sum(),
        2,
    )
)

Total quantity sold: 296663
Unique orders: 122318
Unique customers: 1000
Total gross sales: 55809000.0
Total net sales: 49384617.9
Total profit: 25486012.9


### Preparation overview

In [12]:
preparation = data["fact_preparation"]

print(
    "Total planned quantity:",
    preparation["planned_qty"].sum()
)

print(
    "Total actual preparation:",
    preparation["actual_qty"].sum()
)

print(
    "Planning variance:",
    (
        preparation["actual_qty"].sum()
        - preparation["planned_qty"].sum()
    )
)

Total planned quantity: 311656
Total actual preparation: 319439
Planning variance: 7783


### Inventory overview

In [13]:
inventory = data["fact_inventory"]

print(
    "Total received quantity:",
    inventory["received_qty"].sum()
)

print(
    "Total prepared quantity:",
    inventory["prepared_qty"].sum()
)

print(
    "Total sold quantity:",
    inventory["sold_qty"].sum()
)

print(
    "Stockout records:",
    inventory["stockout_flag"].sum()
)

Total received quantity: 252035
Total prepared quantity: 319439
Total sold quantity: 296663
Stockout records: 0


### Waste overview

In [14]:
waste = data["fact_waste"]

print(
    "Total waste quantity:",
    waste["quantity_wasted"].sum()
)

print(
    "Total waste cost:",
    round(
        waste["waste_cost"].sum(),
        2,
    )
)

print("\nWaste by reason:")

print(
    waste.groupby(
        "waste_reason_id"
    )["quantity_wasted"]
    .sum()
    .sort_values(
        ascending=False
    )
)

Total waste quantity: 11687
Total waste cost: 951885.0

Waste by reason:
waste_reason_id
Overproduction       10981
Expired                578
Damaged                 57
Preparation Error       39
Quality Issue           32
Name: quantity_wasted, dtype: int64


## First business observations

### 3. Initial Observations

The initial data-understanding stage provides a high-level view of
SmartServe's operational structure.

Key areas identified for deeper analysis include:

1. Sales and revenue variation across stores and products.
2. Differences between planned and actual preparation.
3. Inventory availability and stockout patterns.
4. Waste volume and waste cost by product, store, and reason.
5. Relationships between promotions, weather, and demand.
6. Customer purchasing patterns.
7. The suitability of the data for demand forecasting and waste-risk
   prediction.

These observations are descriptive only. Detailed conclusions will be
developed during exploratory analysis, SQL business analysis, and
machine-learning stages.